# 🛡️ AEGIS MARKETS — Professional Quant Training Pipeline v3.0

**Architecture**: Bidirectional LSTM + Self-Attention → suitable for presentation at any professional level.

| Component | Spec |
|---|---|
| Data | 7 years historical OHLCV |
| Technical Indicators | 16 (MACD, Bollinger, ATR, OBV, ADX, EMA, RSI...) |
| Macro Signals | 5 cross-asset (Crude, VIX, Gold, USD/INR, S&P500) |
| Sentiment | VADER NLP (asset-specific news templates) |
| Model | **Bidirectional LSTM + Self-Attention + Dense** |
| Lookback Window | **30 days** |
| Training | EarlyStopping + ReduceLROnPlateau + ModelCheckpoint |
| Evaluation | MAE, RMSE, MAPE, R², Actual vs Predicted plots |
| Export | Asset-specific .h5 + scaler.pkl + feature_config.json |

## Step 1: Install Libraries

In [ ]:
!pip install -q yfinance vaderSentiment tensorflow scikit-learn pandas numpy matplotlib
print('✅ All libraries installed.')

## Step 2: Imports

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pickle
import json
import requests
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime, timedelta
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Bidirectional, Dense, Dropout,
    BatchNormalization, Layer, Lambda, Multiply, Softmax
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from google.colab import files

print(f'✅ TensorFlow version: {tf.__version__}')
print(f'✅ GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## Step 3: Configuration

**Change `TICKER` here to select your asset:**
- `^NSEI` → Nifty 50 (recommended first)
- `BZ=F` → Brent Crude Oil

Run the notebook **twice** — once per asset — to get both models.

In [ ]:
# ═══════════════════════════════════════════
#   CHANGE THIS TO SWITCH ASSET
TICKER = "^NSEI"    # "^NSEI" or "BZ=F"
# ═══════════════════════════════════════════

YEARS_DATA   = 7       # Years of historical data
LOOKBACK     = 30      # Days the model looks back (window size)
EPOCHS       = 150     # Max training epochs (early stopping will cut this)
BATCH_SIZE   = 32
TEST_SPLIT   = 0.20    # 80% train / 20% test

# ── News API Key ──────────────────────────────────────────────
# DO NOT hardcode your key here and don't paste it in chat/share the notebook with it filled in.
# Get a free key at https://newsapi.org (free tier = 100 req/day, but only covers
# the LAST ~30 DAYS of articles — it cannot backfill full 7-year history).
# Safest way in Colab: Colab → 🔑 icon in left sidebar → "Add new secret" → name it NEWS_API_KEY,
# paste your key there, then this cell reads it automatically (never stored in the notebook file).
try:
    from google.colab import userdata
    NEWS_API_KEY = userdata.get('NEWS_API_KEY')
except Exception:
    NEWS_API_KEY = ""   # falls back to synthetic sentiment if not running in Colab / not set

# Auto-derive names
CLEAN_TICKER = TICKER.replace('^', '').replace('=', '')
ASSET_NAME   = 'Nifty 50' if TICKER == '^NSEI' else 'Brent Crude Oil'
CURRENCY     = 'INR' if TICKER == '^NSEI' else 'USD'
MODEL_FILE   = f'lstm_model_{CLEAN_TICKER}.h5'
SCALER_FILE  = f'scaler_{CLEAN_TICKER}.pkl'
TARGET_SCALER_FILE = f'target_scaler_{CLEAN_TICKER}.pkl'
CONFIG_FILE  = f'feature_config_{CLEAN_TICKER}.json'

START_DATE = (datetime.now() - timedelta(days=YEARS_DATA * 365)).strftime('%Y-%m-%d')
END_DATE   = datetime.now().strftime('%Y-%m-%d')

print(f'📊 Asset          : {ASSET_NAME} ({TICKER})')
print(f'📅 Date Range     : {START_DATE}  →  {END_DATE}')
print(f'🔭 Lookback Window: {LOOKBACK} days')
print(f'📰 Live news key  : {"found ✅" if NEWS_API_KEY else "not set — will use synthetic sentiment fallback"}')
print(f'💾 Output Files   : {MODEL_FILE}, {SCALER_FILE}, {CONFIG_FILE}')

## Step 4: Download OHLCV + Macro Cross-Asset Data

Real quant models feed **cross-asset signals** into each other's prediction:
- **Nifty**: uses Brent Crude, USD/INR, India VIX, Gold, S&P 500
- **Crude**: uses USD Index (DXY), Gold, S&P 500, Natural Gas

In [ ]:
def safe_download(ticker, start, end, name):
    try:
        df = yf.download(ticker, start=start, end=end, progress=False)
        if df.empty:
            print(f'  ⚠️  {name}: No data returned')
            return None
        print(f'  ✅ {name:20s}: {len(df)} trading days')
        return df
    except Exception as e:
        print(f'  ❌ {name}: Failed — {e}')
        return None

print(f'\n📥 Downloading {ASSET_NAME} OHLCV data...')
main_df = safe_download(TICKER, START_DATE, END_DATE, ASSET_NAME)
assert main_df is not None, 'Main asset download failed!'

# Keep OHLCV (Open, High, Low, Close, Volume)
main_df = main_df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
main_df.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
main_df.dropna(inplace=True)

print(f'\n📥 Downloading macro cross-asset signals...')
if TICKER == '^NSEI':
    macro_tickers = {
        'USDINR=X': 'USD_INR',
        '^INDIAVIX': 'India_VIX',
        'GC=F':     'Gold',
        'BZ=F':     'Crude_Oil',
        '^GSPC':    'SP500',
    }
else:
    macro_tickers = {
        'DX-Y.NYB': 'DXY',
        'GC=F':     'Gold',
        '^GSPC':    'SP500',
        'NG=F':     'NatGas',
        '^VIX':     'VIX',
    }

macro_frames = {}
for ticker, name in macro_tickers.items():
    df = safe_download(ticker, START_DATE, END_DATE, name)
    if df is not None:
        macro_frames[name] = df['Close'].rename(name)

print(f'\n✅ Total macro signals loaded: {len(macro_frames)}')
print(f'   Main dataset shape: {main_df.shape}')

## Step 5: News Sentiment (Live + Historical Backfill)

Uses **live headlines from News API** for the most recent ~29 days (free-tier limit), so sentiment stays current every time you re-run the notebook.
Older dates (beyond what the free API can serve) are filled with **synthetic, clearly-labeled placeholder headlines** — be upfront about this split if presenting the project.

In [ ]:
analyzer = SentimentIntensityAnalyzer()

news_by_date = {}

# ── Live news for the most recent window (NewsAPI free tier only serves ~last 30 days) ──
LIVE_NEWS_DAYS = 29  # stay under free-tier limit
live_start = (datetime.now() - timedelta(days=LIVE_NEWS_DAYS)).strftime('%Y-%m-%d')

if NEWS_API_KEY.strip():
    print(f'🌐 Fetching LIVE headlines for last {LIVE_NEWS_DAYS} days from News API...')
    query = ('India stock market OR Nifty OR RBI OR geopolitics'
             if TICKER == '^NSEI' else
             'crude oil OR OPEC OR Brent OR energy market')
    total_fetched = 0
    try:
        for page in range(1, 4):  # up to 3 pages = up to 300 articles
            url = (f'https://newsapi.org/v2/everything'
                   f'?q={query}&from={live_start}&to={END_DATE}'
                   f'&language=en&sortBy=publishedAt&pageSize=100&page={page}'
                   f'&apiKey={NEWS_API_KEY}')
            res = requests.get(url, timeout=10).json()
            if res.get('status') != 'ok':
                print(f'  ⚠️  API returned: {res.get("message", "unknown error")}')
                break
            articles = res.get('articles', [])
            if not articles:
                break
            for art in articles:
                d = art['publishedAt'][:10]
                news_by_date.setdefault(d, []).append(art['title'])
                total_fetched += 1
            if len(articles) < 100:
                break
        print(f'  ✅ Fetched {total_fetched} real headlines across {len(news_by_date)} days (last {LIVE_NEWS_DAYS} days = real, up-to-date sentiment)')
    except Exception as e:
        print(f'  ⚠️  News API failed: {e} — falling back to synthetic for all dates')

days_with_live_news = set(news_by_date.keys())

# ── Synthetic generator fills in every date the live API could NOT reach ──
# (older history beyond the free-tier window, or all dates if no key given)
print('🤖 Generating asset-specific synthetic sentiment for remaining historical days...')
if TICKER == '^NSEI':
    pos = [
        'India GDP beats estimates; RBI holds rates as growth accelerates',
        'FII net buyers as India macro data outperforms expectations',
        'India-US trade deal boosts technology and pharma exports',
        'Brent crude falls on OPEC supply surge, India CAD improves significantly',
        'RBI rate cut boosts bank lending, Sensex hits all-time high',
        'India manufacturing PMI surges to 3-year peak on domestic demand',
        'Rupee strengthens as dollar weakens, FII inflows accelerate',
        'India receives record FDI this quarter, infrastructure stocks rally',
        'India inflation cools below 4%, giving RBI room to cut further',
        'Government capex spending at record levels, infra and PSU stocks surge',
    ]
    neg = [
        'US Fed rate hike triggers FII sell-off across emerging markets',
        'Crude oil surge pressures India import bill, rupee falls sharply',
        'India-Pakistan tensions escalate, defense sectors volatile',
        'Global recession fears hit Indian IT and export stocks hard',
        'India VIX spikes to 2-year high as geopolitical risks mount',
        'US 10Y yield surges, FII exit Indian equities for safety of bonds',
        'India inflation above RBI band, rate hike fears sink markets',
        'China-India border standoff revived, border stocks under pressure',
        'Dollar strengthens on safe-haven demand, rupee hits new low',
        'Nifty selloff deepens as global risk-off sentiment hits emerging markets',
    ]
else:
    pos = [
        'OPEC+ agrees to deeper production cuts, Brent crude jumps',
        'Middle East tensions ease, oil supply disruption fears dissipate',
        'China manufacturing activity surges, oil demand outlook improves',
        'US energy demand beats estimates, crude stockpiles fall',
        'Russia-Saudi oil alliance strengthens, price floor established',
        'EIA reports surprise crude inventory drawdown, prices spike',
    ]
    neg = [
        'OPEC+ unexpectedly increases output, oil price collapses',
        'US shale output at record high, global oil oversupply fears mount',
        'Global recession concerns reduce energy demand projections sharply',
        'Iran nuclear deal progress could add 1 million barrels per day',
        'Demand destruction fears spread as China growth stalls',
        'EIA crude inventory build far exceeds estimates, prices fall',
    ]

neutral = [
    'G20 leaders meet for annual economic cooperation summit',
    'Routine shipping patrols reported along key trade routes',
    'Central bank officials outline standard policy review process',
    'Regional infrastructure summit discusses logistics improvements',
]

np.random.seed(42)
synthetic_count = 0
for d in pd.date_range(start=START_DATE, end=END_DATE):
    d_str = d.strftime('%Y-%m-%d')
    if d_str in days_with_live_news:
        continue  # keep the real headlines we already fetched
    r = np.random.rand()
    if r < 0.28:
        h = [np.random.choice(neg)]
    elif r > 0.72:
        h = [np.random.choice(pos)]
    else:
        h = [np.random.choice(neutral)]
    news_by_date[d_str] = h
    synthetic_count += 1
print(f'  ✅ Filled {synthetic_count} days with synthetic sentiment (label these as placeholder, not real news, if asked)')

# Compute daily sentiment score
daily_sent = {d: np.mean([analyzer.polarity_scores(t)['compound'] for t in titles])
              for d, titles in news_by_date.items()}
sentiment_df = pd.DataFrame.from_dict(daily_sent, orient='index', columns=['Sentiment'])
sentiment_df.index = pd.to_datetime(sentiment_df.index)
sentiment_df.sort_index(inplace=True)

print(f'\n✅ Sentiment range: {sentiment_df.Sentiment.min():.3f} to {sentiment_df.Sentiment.max():.3f}')
print(f'   Mean sentiment : {sentiment_df.Sentiment.mean():.3f}')
print(f'   Real news days : {len(days_with_live_news)}  |  Synthetic days: {synthetic_count}')

## Step 6: Professional Feature Engineering (20+ Features)

All computed from scratch using pandas — no extra libraries needed.

| Category | Features |
|---|---|
| Price | Open, High, Low, Close, Volume |
| Trend | SMA10, SMA20, EMA20, EMA50, SMA_ratio |
| Momentum | RSI, MACD, MACD_Signal, MACD_Hist, Momentum_5 |
| Volatility | BB_Upper, BB_Lower, BB_Width, ATR, Rolling_Vol |
| Volume | OBV |
| Sentiment | VADER score |
| Macro | 3–5 cross-asset returns |

In [ ]:
df = main_df.copy()

# ── Trend ──────────────────────────────────────────────────────
df['SMA_10']   = df['Close'].rolling(10).mean()
df['SMA_20']   = df['Close'].rolling(20).mean()
df['EMA_20']   = df['Close'].ewm(span=20, adjust=False).mean()
df['EMA_50']   = df['Close'].ewm(span=50, adjust=False).mean()
df['SMA_Ratio']= df['SMA_10'] / (df['SMA_20'] + 1e-9)  # golden/death cross signal

# ── Momentum ────────────────────────────────────────────────────
delta = df['Close'].diff()
gain  = delta.where(delta > 0, 0).rolling(14).mean()
loss  = (-delta.where(delta < 0, 0)).rolling(14).mean()
df['RSI'] = 100 - (100 / (1 + gain / (loss + 1e-9)))

ema12 = df['Close'].ewm(span=12, adjust=False).mean()
ema26 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD']        = ema12 - ema26
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_Hist']   = df['MACD'] - df['MACD_Signal']
df['Momentum_5']  = df['Close'].pct_change(5)

# ── Volatility ──────────────────────────────────────────────────
std20 = df['Close'].rolling(20).std()
df['BB_Upper'] = df['SMA_20'] + 2 * std20
df['BB_Lower'] = df['SMA_20'] - 2 * std20
df['BB_Width'] = (df['BB_Upper'] - df['BB_Lower']) / (df['SMA_20'] + 1e-9)

tr1 = df['High'] - df['Low']
tr2 = (df['High'] - df['Close'].shift()).abs()
tr3 = (df['Low']  - df['Close'].shift()).abs()
true_range = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
df['ATR'] = true_range.rolling(14).mean()
df['Rolling_Vol'] = df['Close'].pct_change().rolling(10).std()

# ── Volume ──────────────────────────────────────────────────────
obv = [0]
for i in range(1, len(df)):
    if df['Close'].iloc[i] > df['Close'].iloc[i-1]:
        obv.append(obv[-1] + df['Volume'].iloc[i])
    elif df['Close'].iloc[i] < df['Close'].iloc[i-1]:
        obv.append(obv[-1] - df['Volume'].iloc[i])
    else:
        obv.append(obv[-1])
df['OBV'] = obv
df['OBV'] = df['OBV'] / 1e7  # scale down for numerical stability

# ── Sentiment ───────────────────────────────────────────────────
df = df.merge(sentiment_df, left_index=True, right_index=True, how='left')
df['Sentiment'] = df['Sentiment'].fillna(0.0)

# ── Macro Cross-Asset Returns ───────────────────────────────────
macro_feature_names = []
for name, series in macro_frames.items():
    col_name = f'Macro_{name}_5d'
    pct = series.pct_change(5).rename(col_name)
    df = df.merge(pct, left_index=True, right_index=True, how='left')
    df[col_name] = df[col_name].fillna(0.0)
    macro_feature_names.append(col_name)

# ── Final Cleanup ───────────────────────────────────────────────
df.dropna(inplace=True)

# Feature list (Close must be index 0 for inverse-transform to work)
FEATURE_COLS = (
    ['Close', 'Open', 'High', 'Low', 'Volume',
     'SMA_10', 'SMA_20', 'EMA_20', 'EMA_50', 'SMA_Ratio',
     'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'Momentum_5',
     'BB_Upper', 'BB_Lower', 'BB_Width', 'ATR', 'Rolling_Vol',
     'OBV', 'Sentiment']
    + macro_feature_names
)

print(f'✅ Dataset shape after feature engineering: {df.shape}')
print(f'✅ Total features: {len(FEATURE_COLS)}')
print(f'   Features: {FEATURE_COLS}')
df[FEATURE_COLS].tail(3)

## Step 7: Train/Test Split & Scaling

In [ ]:
data_matrix = df[FEATURE_COLS].values.astype(np.float32)
n_features  = data_matrix.shape[1]

# 80/20 split FIRST (on raw values) — test set must stay unseen
split = int(len(data_matrix) * (1 - TEST_SPLIT))
train_raw = data_matrix[:split]
test_raw  = data_matrix[split:]

# Fit scaler ONLY on train data, then transform both — no leakage
scaler = MinMaxScaler(feature_range=(0, 1))
train_data = scaler.fit_transform(train_raw)
test_data  = scaler.transform(test_raw)

print(f'✅ Scaler fitted on TRAIN ONLY | Close price range: {scaler.data_min_[0]:.2f} → {scaler.data_max_[0]:.2f} {CURRENCY}')
print(f'✅ Train: {len(train_data)} rows | Test: {len(test_data)} rows')
print(f'   (Scaler never saw test-set values — no data leakage)')

# Save scaler immediately
with open(SCALER_FILE, 'wb') as f:
    pickle.dump(scaler, f)
print(f'✅ Scaler saved: {SCALER_FILE}')

## Step 8: Build Sequence Datasets (30-Day Lookback)

In [ ]:
from sklearn.preprocessing import StandardScaler

def create_sequences(scaled_data, raw_close, lookback):
    """X = lookback window of scaled features. y = NEXT-DAY RETURN (not price).
    Predicting raw price lets the model 'cheat' by echoing yesterday's close
    (looks accurate on MAPE, but explains none of the actual movement -> negative R2).
    Predicting the return forces the model to actually learn what drives movement."""
    X, y, prev_close = [], [], []
    for i in range(lookback, len(scaled_data)):
        X.append(scaled_data[i - lookback:i, :])
        ret = (raw_close[i] - raw_close[i - 1]) / (raw_close[i - 1] + 1e-9)
        y.append(ret)
        prev_close.append(raw_close[i - 1])
    return (np.array(X, dtype=np.float32),
            np.array(y, dtype=np.float32),
            np.array(prev_close, dtype=np.float32))

X_train, y_train_ret, prev_close_train = create_sequences(train_data, train_raw[:, 0], LOOKBACK)
X_test,  y_test_ret,  prev_close_test  = create_sequences(test_data,  test_raw[:, 0],  LOOKBACK)

# Standardize the return target using TRAIN stats only (same no-leakage rule as before)
target_scaler = StandardScaler()
y_train = target_scaler.fit_transform(y_train_ret.reshape(-1, 1)).flatten().astype(np.float32)
y_test  = target_scaler.transform(y_test_ret.reshape(-1, 1)).flatten().astype(np.float32)

with open(TARGET_SCALER_FILE, 'wb') as f:
    pickle.dump(target_scaler, f)

print(f'✅ X_train: {X_train.shape}  (Samples x Timesteps x Features)')
print(f'✅ X_test : {X_test.shape}')
print(f'✅ Target : next-day RETURN (standardized), not raw price')
print(f'   Train return range: {y_train_ret.min()*100:.2f}% to {y_train_ret.max()*100:.2f}%')
print(f'\nModel will learn from {LOOKBACK} days of history to predict next-day % change.')

## Step 9: Build Bidirectional LSTM + Self-Attention Model

```
Input (30 days × N features)
      │
  BiLSTM(128)  ← learns both past→future and future→past patterns
      │
  BatchNorm + Dropout(0.3)
      │
  BiLSTM(64, return_sequences=True)
      │
  Self-Attention  ← model focuses on the most important timesteps
      │
  BiLSTM(32)
      │
  BatchNorm + Dropout(0.2)
      │
  Dense(64, relu) → Dense(32, relu) → Dense(1)
      │
  Next-Day Close Price (scaled)
```

In [ ]:
class SelfAttention(Layer):
    """Additive Self-Attention — learns which timesteps matter most."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name='attn_weight', shape=(input_shape[-1], 1),
            initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(
            name='attn_bias', shape=(input_shape[1], 1),
            initializer='zeros', trainable=True)
        super().build(input_shape)

    def call(self, x):
        e = tf.nn.tanh(tf.matmul(x, self.W) + self.b)   # (batch, T, 1)
        a = tf.nn.softmax(e, axis=1)                      # attention weights
        context = tf.reduce_sum(x * a, axis=1)            # weighted sum
        return context

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

def build_model(lookback, n_features):
    inp = Input(shape=(lookback, n_features), name='price_input')

    # ── Layer 1: BiLSTM — captures long-range patterns in both directions ──
    x = Bidirectional(LSTM(128, return_sequences=True), name='bilstm_1')(inp)
    x = BatchNormalization()(x)
    x = Dropout(0.30)(x)

    # ── Layer 2: BiLSTM + Self-Attention — focus on most informative days ──
    x = Bidirectional(LSTM(64, return_sequences=True), name='bilstm_2')(x)
    x = BatchNormalization()(x)
    x = SelfAttention(name='self_attention')(x)
    x = Dropout(0.20)(x)

    # ── Dense head ─────────────────────────────────────────────────────────
    x = Dense(64, activation='relu', name='dense_1')(x)
    x = Dropout(0.15)(x)
    x = Dense(32, activation='relu', name='dense_2')(x)
    out = Dense(1, name='output')(x)

    model = Model(inp, out)
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='huber',   # Huber loss: less sensitive to outlier spikes
        metrics=['mae']
    )
    return model

model = build_model(LOOKBACK, n_features)
model.summary()

total_params = model.count_params()
print(f'\n✅ Model built | Total parameters: {total_params:,}')

## Step 10: Train with Smart Callbacks

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        MODEL_FILE,
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
]

print(f'🚀 Training Bidirectional LSTM + Attention on {ASSET_NAME}...')
print(f'   Samples: {len(X_train)} train | {len(X_test)} test')
print(f'   Features: {n_features} | Lookback: {LOOKBACK} days')
print(f'   Max epochs: {EPOCHS} (early stopping after 15 no-improvement)\n')

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
    verbose=1
)

best_epoch = np.argmin(history.history['val_loss']) + 1
best_val_loss = min(history.history['val_loss'])
print(f'\n✅ Training complete!')
print(f'   Best epoch    : {best_epoch}')
print(f'   Best val_loss : {best_val_loss:.6f}')

## Step 11: Training Curve Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(f'Aegis Markets Training Curves — {ASSET_NAME}', fontsize=14, fontweight='bold')

# Loss curve
axes[0].plot(history.history['loss'],     label='Train Loss', color='#3b82f6', lw=2)
axes[0].plot(history.history['val_loss'], label='Val Loss',   color='#f59e0b', lw=2)
axes[0].axvline(best_epoch - 1, color='#10b981', ls='--', lw=1.5, label=f'Best Epoch ({best_epoch})')
axes[0].set_title('Huber Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE curve
axes[1].plot(history.history['mae'],     label='Train MAE', color='#3b82f6', lw=2)
axes[1].plot(history.history['val_mae'], label='Val MAE',   color='#f59e0b', lw=2)
axes[1].set_title('Mean Absolute Error')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE (scaled)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved.')

## Step 12: Professional Evaluation — MAE, RMSE, MAPE, R²

These are the **4 metrics** used to judge any regression model professionally.

In [ ]:
print('📊 Evaluating model on test set...')
pred_ret_scaled = model.predict(X_test, verbose=0)

# Inverse-transform the RETURN prediction (not price this time)
pred_returns = target_scaler.inverse_transform(pred_ret_scaled).flatten()
actual_returns = y_test_ret

# Reconstruct actual prices from returns: price = prev_close * (1 + return)
predictions = prev_close_test * (1 + pred_returns)
actuals     = prev_close_test * (1 + actual_returns)

# ── Price-level Metrics ────────────────────────────────────────
mae  = mean_absolute_error(actuals, predictions)
rmse = np.sqrt(mean_squared_error(actuals, predictions))
mape = np.mean(np.abs((actuals - predictions) / (np.abs(actuals) + 1e-9))) * 100
r2   = r2_score(actuals, predictions)

# ── Return-level Metrics (the ones that actually matter for a return-based model) ──
return_r2 = r2_score(actual_returns, pred_returns)
directional_accuracy = np.mean(np.sign(pred_returns) == np.sign(actual_returns)) * 100

# Naive baseline: predict tomorrow = today (dumbest possible model)
baseline_pred = actuals[:-1]
baseline_act  = actuals[1:]
baseline_mape = np.mean(np.abs((baseline_act - baseline_pred) / (np.abs(baseline_act) + 1e-9))) * 100

print('\n' + '═' * 55)
print(f'  📊  MODEL EVALUATION — {ASSET_NAME} ({CURRENCY})')
print('═' * 55)
print(f'  MAE  (price)                 : {mae:>10.4f} {CURRENCY}')
print(f'  RMSE (price)                 : {rmse:>10.4f} {CURRENCY}')
print(f'  MAPE (price)                 : {mape:>10.4f} %')
print(f'  R²   (price level)           : {r2:>10.4f}')
print('─' * 55)
print(f'  R²   (RETURN — the real test): {return_r2:>10.4f}')
print(f'  Directional Accuracy         : {directional_accuracy:>9.2f} %   (50% = random guess)')
print('─' * 55)
print(f'  Naive Baseline MAPE          : {baseline_mape:>10.4f} %')
improvement = baseline_mape - mape
verdict = '✅ BEATS baseline' if improvement > 0 else '❌ Below baseline'
print(f'  LSTM vs Baseline             : {improvement:>+10.4f} pp  {verdict}')
print('═' * 55)

print('\n  How to read this:')
print('  - Price-level MAPE/R² can look good even for a bad model (echo effect).')
print('  - Return R² and Directional Accuracy are the honest scores for a return-based model.')
print('  - Directional Accuracy > 52-55% on real market data is considered genuinely useful;')
print('    beating 50% consistently out-of-sample is already hard.')

if directional_accuracy >= 55:
    grade = 'Strong — meaningfully beats random guessing'
elif directional_accuracy >= 52:
    grade = 'Modest edge — better than random but small'
elif directional_accuracy >= 48:
    grade = 'No real edge — statistically close to a coin flip'
else:
    grade = 'Worse than random — check for a bug before trusting this'
print(f'\n  Verdict: {grade} ({directional_accuracy:.1f}% directional accuracy)')

## Step 13: Actual vs Predicted Plot + Residual Analysis

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig)
fig.suptitle(f'Aegis Markets v3.0 — {ASSET_NAME} Model Evaluation', fontsize=14, fontweight='bold')

# 1. Actual vs Predicted (full test set)
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(actuals,     label='Actual Price',    color='#3b82f6', lw=2)
ax1.plot(predictions, label='Predicted Price', color='#f59e0b', lw=1.5, ls='--')
ax1.fill_between(range(len(actuals)), actuals, predictions, alpha=0.08, color='#9ca3af')
ax1.set_title(f'Actual vs Predicted Close — Test Set ({int(TEST_SPLIT*100)}%)')
ax1.set_xlabel('Trading Days')
ax1.set_ylabel(f'Price ({CURRENCY})')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Scatter: Actual vs Predicted
ax2 = fig.add_subplot(gs[1, 0])
ax2.scatter(actuals, predictions, alpha=0.4, s=8, color='#3b82f6')
mn, mx = actuals.min(), actuals.max()
ax2.plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect Prediction')
ax2.set_title(f'Scatter: Actual vs Predicted  (R²={r2:.4f})')
ax2.set_xlabel(f'Actual ({CURRENCY})')
ax2.set_ylabel(f'Predicted ({CURRENCY})')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Residuals distribution
ax3 = fig.add_subplot(gs[1, 1])
residuals = actuals - predictions
ax3.hist(residuals, bins=40, color='#f59e0b', edgecolor='white', alpha=0.8)
ax3.axvline(0, color='#ef4444', lw=2, ls='--')
ax3.set_title(f'Residual Distribution  (MAE={mae:.2f} {CURRENCY})')
ax3.set_xlabel(f'Error ({CURRENCY})')
ax3.set_ylabel('Frequency')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Evaluation plots saved.')

## Step 14: Save Model + Feature Config + Download

The `feature_config.json` file tells the dashboard **exactly** which features and lookback window this model uses — so it loads correctly automatically.

In [ ]:
# Save feature configuration JSON (tells app.py how to reconstruct features)
config = {
    'ticker':      TICKER,
    'asset_name':  ASSET_NAME,
    'currency':    CURRENCY,
    'features':    FEATURE_COLS,
    'n_features':  n_features,
    'lookback':    LOOKBACK,
    'macro_used':  list(macro_frames.keys()),
    'trained_on':  END_DATE,
    'target':      'next_day_return',   # model predicts % change, not raw price
    'mape':        round(float(mape), 4),
    'mae':         round(float(mae),  4),
    'rmse':        round(float(rmse), 4),
    'r2_price':    round(float(r2),   4),
    'r2_return':   round(float(return_r2), 4),
    'directional_accuracy': round(float(directional_accuracy), 2),
    'close_min':   float(scaler.data_min_[0]),
    'close_max':   float(scaler.data_max_[0]),
}

with open(CONFIG_FILE, 'w') as f:
    json.dump(config, f, indent=2)

print('✅ Files ready for download:')
print(f'   {MODEL_FILE:35s} <- trained BiLSTM+Attention model')
print(f'   {SCALER_FILE:35s} <- MinMaxScaler fitted to feature range')
print(f'   {TARGET_SCALER_FILE:35s} <- StandardScaler fitted to return range')
print(f'   {CONFIG_FILE:35s} <- feature list + evaluation metrics')
print(f'\n📊 Model Summary:')
print(f'   MAPE (price):            {mape:.4f}%')
print(f'   R² (return, honest one): {return_r2:.4f}')
print(f'   Directional Accuracy:    {directional_accuracy:.2f}%')
print(f'   Features: {n_features}')
print(f'   Lookback: {LOOKBACK} days')

print('\n⬇️  Downloading all 4 files...')
files.download(MODEL_FILE)
files.download(SCALER_FILE)
files.download(TARGET_SCALER_FILE)
files.download(CONFIG_FILE)
print('✅ Done! Place all 4 files together in your Streamlit project folder.')

---
## 📌 After Training — What to Do

1. **Download** all 3 files per asset (run notebook twice — once for Nifty, once for Crude)
2. **Place** files in your project folder
3. **Refresh** the Streamlit app at `http://localhost:8501`
4. The app auto-reads `feature_config_NSEI.json` to know which features and lookback to use

## 📌 How to Explain This Model in an Interview

- **Data**: 7 years historical OHLCV from Yahoo Finance + 5 macro cross-asset signals
- **Features**: 22–27 features including RSI, MACD, Bollinger Bands, ATR, OBV, news sentiment
- **Sentiment**: live News API headlines for the last ~29 days, synthetic placeholder sentiment for older history — be honest about this if asked, it's a realistic constraint of free news APIs
- **Model**: Bidirectional LSTM captures patterns in both temporal directions; Self-Attention identifies which of the 30 input days carry the most predictive weight
- **Training**: Huber loss (robust to price outliers), EarlyStopping, ReduceLROnPlateau
- **Scaling**: MinMaxScaler fit ONLY on train data, applied to test data — avoids data leakage
- **Evaluation**: MAE, RMSE, MAPE, R² on held-out 20% test set; compared against naive baseline
- **Production**: Deployed in a Streamlit dashboard with live yfinance data, VADER NLP, and Gemini AI chatbot